In [8]:
import spotipy
import json
import os
import sys
from datetime import datetime
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyOAuth

# Garante que o Python encontre a pasta 'utils' na raiz do projeto
sys.path.append(os.path.abspath(os.path.join('..')))
from utils.minio_client import MinioClient

Conexão com a API:

In [9]:
load_dotenv()
# escopo de endpoints da API - top melhores músicas, últimas músicas tocadas, leitura à biblioteca do usuário
scope = "user-top-read, user-read-recently-played"
# autenticação com a api
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv("APP_CLIENT_ID"),
    client_secret=os.getenv("APP_CLIENT_SECRET"),
    redirect_uri=os.getenv("APP_REDIRECT_URI"),
    scope=scope,
    cache_path=".cache",
    #open_browser= False
    ))

Extração dos dados da API:

In [10]:
# chamadas da api, top artistas, músicas, ultimas musicas ouvidas, albums salvos e utimas playlists ouvidas
artists = sp.current_user_top_artists(limit=50, offset=0, time_range="long_term")
tracks = sp.current_user_top_tracks(limit=50, offset=0, time_range="long_term")
ultimas_ouvidas = sp.current_user_recently_played(limit=50)

# instancias minio/local paths
minio = MinioClient()
bucket_name = os.getenv("BUCKET_NAME")
base_path = "../data/raw"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# O dicionário define a 'chave' (nome da pasta) e o 'conteúdo' (os dados do Spotify)
dados_para_salvar = {
    "artists": artists,
    "tracks": tracks,
    "recently_played": ultimas_ouvidas,
}

for folder_name, data in dados_para_salvar.items():

    # Caminho local:
    local_dir = os.path.join(base_path, folder_name)
    os.makedirs(local_dir, exist_ok=True)
    
    # Nome do arquivo local
    file_name = f"{folder_name}_{timestamp}.json"
    local_file_path = os.path.join(local_dir, file_name)

    # Salva o JSON localmente
    with open(local_file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
        
    # Upload Bronze MinIO
    s3_object_name = f"bronze/{folder_name}/{file_name}"
    
    if minio.upload_file(bucket_name, s3_object_name, local_file_path):
        print(f"{folder_name.upper()} salvo: {bucket_name}/{s3_object_name}")

ARTISTS salvo: spotify-data-pipeline/bronze/artists/artists_20260424_151010.json
TRACKS salvo: spotify-data-pipeline/bronze/tracks/tracks_20260424_151010.json
RECENTLY_PLAYED salvo: spotify-data-pipeline/bronze/recently_played/recently_played_20260424_151010.json
